In [14]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score

# --- load the saved A4 state ---
train = pd.read_parquet(Path("../data/interim/train_a4.parquet"))
SPEC = json.load(open("../reports/a4_spec.json"))
FIT_CUTOFF = pd.Timestamp("2014-12-31")
TARGET = "defaulted"

# --- application_type: constant in fit ---
SPEC["exclude"]["application_type"] = ("constant in the fit sample (Individual only); joint applications appear "
                                       "from 2015 and in the holdout: unseen by the model, route to review / monitor")
SPEC["features"] = [c for c in SPEC["features"] if c != "application_type"]

# --- emp_length -> years, purpose -> groups (>= 100 loans in fit) ---
def emp_length_years(s):
    return s.replace("< 1 year", "0 years").str.extract(r"(\d+)", expand=False).astype(float)

def fit_kept_levels(s, min_loans=100):
    counts = s.value_counts()
    return sorted(counts[counts >= min_loans].index.tolist())

def apply_levels(s, keep, other="other"):
    return s.where(s.isin(keep), other)

is_fit = train["issue_dt"] <= FIT_CUTOFF
PURPOSE_KEEP = fit_kept_levels(train.loc[is_fit, "purpose"])
train["emp_length_years"] = emp_length_years(train["emp_length"])
train["purpose_grp"] = apply_levels(train["purpose"], PURPOSE_KEEP)
SPEC["purpose_keep"] = PURPOSE_KEEP
SPEC["exclude"]["emp_length"] = "replaced by emp_length_years (0-10, NaN = unknown)"
SPEC["exclude"]["purpose"] = "replaced by purpose_grp (levels < 100 loans in fit merged into 'other')"
SPEC["features"] = sorted((set(SPEC["features"]) - {"emp_length", "purpose"}) | {"emp_length_years", "purpose_grp"})

# --- emp_title -> job_family (final keyword rules, first match wins) ---
JOB_FAMILIES = [
    ("owner_self", r"\b(?:owner|self employed|self|proprietor|founder|partner)\b"),
    ("healthcare", r"\b(?:nurse|rn|lpn|cna|physician|doctor|dentist|dental|hygienist|pharmac\w*|therapist|medical|hospital|clinic\w*|health\w*|paramedic|emt|surgical|radiolog\w*)\b"),
    ("education", r"\b(?:teacher|professor|instructor|educator|school|university|college|tutor|principal|faculty)\b"),
    ("legal_social", r"\b(?:attorney|lawyer|paralegal|legal|counsel\w*|social worker|case manager|pastor|minister|clergy)\b"),
    ("public_safety_gov", r"\b(?:police|sheriff|deputy|firefighter|army|navy|air force|military|usps|postal|letter carrier|sergeant|correction\w*|federal|county|city of|state of)\b"),
    ("tech_engineering", r"\b(?:engineer\w*|developer|programmer|software|it|analyst|architect|scientist|network|systems?|data)\b"),
    ("transport_trades", r"\b(?:driver|truck\w*|technician|tech|mechanic|electrician|operator|machinist|welder|carpenter|plumber|painter|maintenance|installer|warehouse|laborer|foreman|construction|production|assembler|forklift)\b"),
    ("finance_admin", r"\b(?:account\w*|bookkeeper|controller|finance|financial|bank\w*|teller|underwriter|auditor|insurance|adjuster|admin\w*|assistant|clerk|secretary|receptionist|office|coordinator|specialist|representative|rep)\b"),
    ("sales_service", r"\b(?:sales|cashier|server|retail|store|customer|service|agent|attendant|cook|chef|bartender|waitress|stylist|associate|merchandiser)\b"),
    ("management", r"\b(?:manager|management|mgr|director|supervisor|president|vp|ceo|cfo|coo|chief|executive|lead|leader|head|superintendent)\b"),
]

def normalize_title(s):
    return (s.fillna("").str.lower()
             .str.replace(r"[^a-z ]", " ", regex=True)
             .str.replace(r"\s+", " ", regex=True).str.strip())

def job_family(s):
    text = normalize_title(s)
    out = pd.Series("other", index=s.index)
    assigned = pd.Series(False, index=s.index)
    for family, pattern in JOB_FAMILIES:
        hit = text.str.contains(pattern, regex=True) & ~assigned
        out[hit] = family
        assigned |= hit
    out[s.isna()] = "unknown"
    return out

train["job_family"] = job_family(train["emp_title"])
SPEC["job_families"] = [[f, p] for f, p in JOB_FAMILIES]
SPEC["exclude"]["emp_title"] = "replaced by job_family (keyword rules, first match wins; 65.5% matched)"
SPEC["pending"].pop("emp_title", None)
SPEC["features"] = sorted(set(SPEC["features"]) | {"job_family"})

# --- verification: Verified + Source Verified merged ---
VERIF_MAP = {"Verified": "verified", "Source Verified": "verified", "Not Verified": "not_verified"}
train["verification_grp"] = train["verification_status"].map(VERIF_MAP)
SPEC["verification_map"] = VERIF_MAP
SPEC["exclude"]["verification_status"] = ("replaced by verification_grp: Verified and Source Verified merged "
                                          "(WoE -0.138 vs -0.118; IV 0.0458 -> 0.0457)")
SPEC["features"] = sorted((set(SPEC["features"]) - {"verification_status"}) | {"verification_grp"})

# --- purpose: 3 risk levels for the scorecard ---
PURPOSE_RISK = {"small_business": "high",
                "debt_consolidation": "medium", "other": "medium", "medical": "medium",
                "credit_card": "low", "home_improvement": "low", "major_purchase": "low", "car": "low"}
train["purpose_risk"] = train["purpose_grp"].map(PURPOSE_RISK)
SPEC.setdefault("scorecard_coarse_classing", {})["purpose_grp"] = PURPOSE_RISK
SPEC["notes"] = SPEC.get("notes", []) + [
    "purpose: 3 risk levels for the scorecard. Inside-level chi-square p = 0.855 / 0.942, between p = 4e-10; "
    "2015 AUC with 2013-14 WoE 0.5415 vs 0.5443 for 8 groups (difference < 0.005, within noise). "
    "Monitor medical (20.0% -> 29.5%, n~150/yr).",
]

# --- helper functions (final versions) ---
def level_table(df, col, target=TARGET):
    g = df.groupby(col, dropna=False)[target]
    t = pd.DataFrame({"loans": g.size(), "share": (g.size() / len(df)).round(3), "default_rate": g.mean().round(3)})
    return t.sort_values("loans", ascending=False)

def woe_iv(df, col, target=TARGET, n_bins=10):
    s = df[col]
    if not pd.api.types.is_numeric_dtype(s) or s.nunique() <= 12:
        groups = s.astype(str).where(s.notna(), "missing")
    else:
        groups = pd.qcut(s, n_bins, labels=False, duplicates="drop").fillna(-1).astype(int)
    t = pd.crosstab(groups, df[target])
    good, bad = t[0] + 0.5, t[1] + 0.5
    pct_good, pct_bad = good / good.sum(), bad / bad.sum()
    woe = np.log(pct_good / pct_bad)
    table = pd.DataFrame({"loans": t.sum(axis=1), "default_rate": (t[1] / t.sum(axis=1)).round(3), "woe": woe.round(3)})
    return table, float(((pct_good - pct_bad) * woe).sum())

def woe_map(df, col):
    return woe_iv(df, col)[0]["woe"].to_dict()

def out_of_time_auc(fit_df, later_df, col):
    s = later_df[col]
    groups = s.astype(str).where(s.notna(), "missing")
    woe = groups.map(woe_map(fit_df, col)).fillna(0.0)
    return round(roc_auc_score(later_df[TARGET], -woe), 4)

fit = train[is_fit].copy()
later = train[~is_fit].copy()

# --- checks: these must match yesterday's results ---
print("features:", len(SPEC["features"]))                                         # expect 31
print("purpose_grp AUC:", out_of_time_auc(fit, later, "purpose_grp"))              # expect 0.5443
print("job_family AUC:", out_of_time_auc(fit, later, "job_family"))                # expect 0.5438
print("emp_length_years AUC:", out_of_time_auc(fit, later, "emp_length_years"))    # expect 0.5076

features: 31
purpose_grp AUC: 0.5443
job_family AUC: 0.5438
emp_length_years AUC: 0.5076


In [15]:
SPEC["exclude"]["emp_length_years"] = ("no predictive value on new data: 2015 AUC 0.5076 with 2013-14 WoE "
                                       "(rule: < 0.53 -> exclude); IV 0.012")
SPEC["features"] = [c for c in SPEC["features"] if c != "emp_length_years"]

SPEC["fairness_review"] = {
    "job_family": ("weak but real signal (2015 AUC 0.5438, IV 0.014); possible proxy for protected traits. "
                   "Keep only if it adds >= 0.005 validation AUC to the full model (Part B ablation); "
                   "excluded from the scorecard by default."),
}
print("features:", len(SPEC["features"]), "| fairness review:", list(SPEC["fairness_review"]))   # expect 30 | ['job_family']

features: 30 | fairness review: ['job_family']


In [16]:
with open("../reports/a5_spec.json", "w") as f:
    json.dump(SPEC, f, indent=2)
train.to_parquet("../data/interim/train_a5.parquet", index=False)

In [17]:
def woe_with_noise(df, col):
    """WoE table plus a rough 95% uncertainty (±) of each group's default rate."""
    table = woe_iv(df, col)[0]
    p, n = table["default_rate"], table["loans"]
    table["plus_minus"] = (2 * np.sqrt(p * (1 - p) / n)).round(3)
    return table.sort_values("woe")

display(woe_with_noise(fit, "home_ownership"))
print("levels in 2013-14:", sorted(fit["home_ownership"].unique()))
print("levels in 2015:   ", later["home_ownership"].value_counts().to_dict())

,loans,default_rate,woe,plus_minus
home_ownership,,,,
RENT,5560,0.198,-0.170,0.011
OWN,1406,0.171,0.011,0.020
MORTGAGE,7407,0.154,0.139,0.008


levels in 2013-14: ['MORTGAGE', 'OWN', 'RENT']
levels in 2015:    {'MORTGAGE': 7558, 'RENT': 6361, 'OWN': 1708}


In [18]:
SPEC["unseen_category_rule"] = "any category level not seen in the fit sample gets WoE 0 (neutral) in the scorecard"
SPEC["notes"].append("home_ownership: kept as 3 levels (RENT WoE -0.170, OWN 0.011, MORTGAGE 0.139; "
                     "all groups >= 1,406 loans; no new levels in 2015)")
with open("../reports/a5_spec.json", "w") as f:
    json.dump(SPEC, f, indent=2)
print("saved | features:", len(SPEC["features"]))          # expect 30

saved | features: 30
